# 07 - AI 智能配置

> **何时使用**: 当你不想手写 YAML 配置，希望 AI 自动分析表结构并生成最优配置时。
>
> **核心概念**: `sqlseed-ai` 插件使用 LLM 分析 schema 语义，自动生成配置，带自纠正循环。

## 适用场景

- 表结构复杂，不想手写配置 → AI 自动生成
- 不确定该用哪个生成器 → AI 根据列名语义推荐
- 需要快速原型 → AI 一行命令生成配置

## 你将学到

- SchemaAnalyzer 工作流程
- AiConfigRefiner 自纠正循环
- 模型自动选择与回退
- ErrorSummary 错误分类
- 缓存机制

⚠️ 需要 API Key（参考 `.env.example`）

**📚 教程导航**

| 序号 | 主题 | 架构层 | 前置要求 |
|------|------|--------|----------|
| 01 | 快速上手与核心流程 | Orchestrator | 无 |
| 02 | 9 级策略链详解 | Core: ColumnMapper | 01 |
| 03 | 生成器与 Provider 体系 | Generators | 01 |
| 04 | 数据库层与多表关联 | Database + Core | 01 |
| 05 | 表达式派生与约束求解 | Core: DAG / Expression | 01 |
| 06 | 配置驱动与 Transform | Config / Core | 01 |
| **→ 07** | **AI 智能配置** | **Plugins: AI** | **01** |
| 08 | MCP 服务器集成 | Plugins: MCP | 07 |
| 09 | 插件系统与 Hook 生命周期 | Plugins | 01 |
| 10 | CLI 参考手册 | CLI | 06 |
| 11 | 工具类参考 | Utils | 01 |
| 12 | 测试集成模式 | Testing | 01 |

---

In [1]:
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

c:\Users\14435\Desktop\sqlseed\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating organizations: 100%|██████████| 5/5 [00:00<00:00, 176.33it/s]
2026-06-13T05:48:26.052662Z [warning  ] AI API key not configured. Set GOOGLE_API_KEY, SQLSEED_AI_API_KEY, or OPENAI_API_KEY environment variable. For Ollama, set SQLSEED_AI_BACKEND=ollama.
Generating members: 100%|██████████| 20/20 [00:00<00:00, 279.26it/s]
2026-06-13T05:48:26.180159Z [warning  ] AI API key not configured. Set GOOGLE_API_KEY, SQLSEED_AI_API_KEY, or OPENAI_API_KEY environment variable. For Ollama, set SQLSEED_AI_BACKEND=ollama.
Generating tags: 100%|██████████| 8/8 [00:00<00:00, 142.17it/s]


sqlseed 0.2.2.dev1+g7f73669bf.d20260609 | Database: C:\Users\14435\Desktop\sqlseed\examples\sqlseed_demo.db


### 📍 架构定位

| 模块 | 文件 | 核心类/函数 |
|------|------|------------|
| Schema 分析 | `plugins/sqlseed-ai/src/sqlseed_ai/analyzer.py` | `SchemaAnalyzer` |

> 对应架构图: [§7 AI 插件架构](../docs/architecture.zh-CN.md#7-ai-插件架构)

## 1. 先看效果 — AI 自动分析表结构

给 sqlseed-ai 一个表名，它会自动分析 schema、生成最优配置 — **无需手写 YAML**：

```
输入: organizations 表
输出: 自动生成的 YAML 配置（含 generator、params、约束）
```

下面先验证插件安装，再演示完整工作流。

## 2. sqlseed-ai 安装验证

In [2]:
try:
    import sqlseed_ai  # noqa: F401
    print("✅ sqlseed-ai 已安装")
except ImportError:
    print("⚠️ sqlseed-ai 未安装")
    print("   安装: pip install -e ./plugins/sqlseed-ai")

✅ sqlseed-ai 已安装


## 3. SchemaAnalyzer 工作流程

SchemaAnalyzer 收集表的完整上下文（列、索引、FK、样本数据），发送给 LLM 分析后返回 YAML 配置。

In [3]:
from sqlseed_ai.analyzer import SchemaAnalyzer

from sqlseed import connect

with connect(str(db_path)) as orch:
    schema_ctx = orch.get_schema_context('organizations')
    print('Schema context keys:', list(schema_ctx.keys()))
    print(f'Columns: {len(schema_ctx["columns"])}')
    print(f'Foreign keys: {len(schema_ctx["foreign_keys"])}')
    print(f'Indexes: {len(schema_ctx["indexes"])}')
    print(f'Sample data: {len(schema_ctx["sample_data"])} rows')
    print(f'All tables: {schema_ctx["all_table_names"]}')

Schema context keys: ['table_name', 'columns', 'foreign_keys', 'indexes', 'sample_data', 'all_table_names', 'distribution']
Columns: 7
Foreign keys: 1
Indexes: 1
Sample data: 5 rows
All tables: ['organizations', 'members', 'sqlite_sequence', 'projects', 'tasks', 'reviews', 'tags', 'task_tags', 'attachments']


## 4. AiConfigRefiner 自纠正循环

AiConfigRefiner 实现闭环：生成 → 验证 → 修复 → 再验证。如果没有 API Key，展示示例输出。

In [4]:
ORG_PATTERN = r'ORG-\d{4}'
from sqlseed_ai.config import AIConfig, AIBackend

# 使用 AIConfig.from_env() 自动检测后端（支持 LM Studio / Ollama / Google AI Studio）
# 环境变量: SQLSEED_AI_BACKEND, SQLSEED_AI_BASE_URL, SQLSEED_AI_MODEL
ai_config = AIConfig.from_env()

# 检查后端是否可用（LM Studio/Ollama 不需要 API Key，云后端需要）
has_backend = ai_config.backend in (AIBackend.LM_STUDIO, AIBackend.OLLAMA) or ai_config.has_real_api_key

if has_backend:
    from sqlseed_ai.analyzer import SchemaAnalyzer
    from sqlseed_ai.refiner import AiConfigRefiner

    print(f'后端: {ai_config.backend.value}')
    print(f'模型: {ai_config.resolve_model()}')
    try:
        print(f'Base URL: {ai_config.resolve_base_url()}')
    except ValueError as e:
        print(f'配置错误: {e}')
    print()

    analyzer = SchemaAnalyzer(config=ai_config)
    refiner = AiConfigRefiner(analyzer, db_path=str(db_path))

    result = refiner.generate_and_refine('organizations')
    if isinstance(result, dict):
        print(f'AI generated config for {len(result)} columns:')
        for col_name, col_config in result.items():
            if isinstance(col_config, dict):
                print(f'  {col_name}: generator={col_config.get("generator", "N/A")}, params={col_config.get("params", {})}')  # noqa: E501
            else:
                print(f'  {col_name}: {col_config}')
    else:
        print('AI generated config (raw):')
        print(str(result)[:500])
else:
    print('未检测到可用的 AI 后端。请设置以下环境变量之一：')
    print('  - SQLSEED_AI_API_KEY (Google AI Studio / OpenRouter)')
    print('  - SQLSEED_AI_BACKEND=lm_studio (LM Studio 本地推理)')
    print('  - SQLSEED_AI_BACKEND=ollama (Ollama 本地推理)')
    print()
    print('显示示例 AI 输出：')
    print()
    example_config = {
        'org_code': {'generator': 'pattern', 'params': {'pattern': ORG_PATTERN}},
        'name': {'generator': 'company'},
        'parent_code': {'generator': 'pattern', 'params': {'pattern': ORG_PATTERN}},
        'description': {'generator': 'sentence', 'params': {'nb_words': 8}},
        'is_active': {'generator': 'boolean'},
        'member_count': {'generator': 'integer', 'params': {'min_value': 1, 'max_value': 500}},
    }
    for col_name, col_config in example_config.items():
        print(f'  {col_name}: generator={col_config["generator"]}, params={col_config.get("params", {})}')



AI generated config for 3 columns:
  name: organizations
  count: 1000
  columns: [{'name': 'org_code', 'generator': 'pattern', 'params': {'regex': 'ORG-[0-9]{4,6}'}}, {'name': 'name', 'generator': 'company'}, {'name': 'parent_code', 'generator': 'foreign_key', 'params': {'ref_table': 'organizations', 'ref_column': 'org_code'}}, {'name': 'description', 'generator': 'text', 'params': {'min_length': 50, 'max_length': 200}}, {'name': 'created_at', 'generator': 'datetime', 'params': {'start_year': 2000, 'end_year': 2025}}]


## 5. 模型自动选择

`select_best_model` 按优先级尝试免费模型，自动回退。

In [5]:
from sqlseed_ai._model_selector import _GEMMA_MODEL_PRIORITY

print(f'Gemma 4 模型优先级 ({len(_GEMMA_MODEL_PRIORITY)} 个):')
for i, model in enumerate(_GEMMA_MODEL_PRIORITY, 1):
    print(f'  {i}. {model.value} ({model.display_name})')

print('\nselect_gemma_model() 按优先级尝试, 自动回退')



Gemma 4 模型优先级 (5 个):
  1. gemma-4-31b-it (Gemma 4 31B Dense)
  2. gemma-4-26b-a4b-it (Gemma 4 26B A4B MoE (Recommended))
  3. gemma-4-12b-it (Gemma 4 12B Unified (Laptop))
  4. gemma-4-e4b-it (Gemma 4 E4B (4B Effective, Edge))
  5. gemma-4-e2b-it (Gemma 4 E2B (2B Effective, Edge))

select_gemma_model() 按优先级尝试, 自动回退


## 6. ErrorSummary 错误分类

`summarize_error` 将异常分为 7 种类型，用于 AI 自纠正循环。

In [6]:
from sqlseed_ai.errors import summarize_error

test_errors = [
    ValueError('count must be greater than 0'),
    TypeError("'NoneType' object is not iterable"),
    KeyError('missing_column'),
    ImportError('No module named faker'),
    FileNotFoundError('/nonexistent.db'),
    PermissionError('read-only database'),
    RuntimeError('unexpected error'),
]

print('ErrorSummary 7 种错误分类:')
for err in test_errors:
    summary = summarize_error(err)
    print(f'  {type(err).__name__:20s} -> {summary.error_type}')



ErrorSummary 7 种错误分类:
  ValueError           -> runtime_error
  TypeError            -> runtime_error
  KeyError             -> runtime_error
  ImportError          -> runtime_error
  FileNotFoundError    -> fatal
  PermissionError      -> fatal
  RuntimeError         -> runtime_error


## 7. 端到端工作流

完整流程：schema 分析 → AI 生成配置 → 验证 → 填充数据。

In [7]:
ORG_PATTERN = r'ORG-\d{4}'
from pathlib import Path

from sqlseed import fill_from_config
from sqlseed.config.loader import save_config
from sqlseed.config.models import ColumnConfig, GeneratorConfig, TableConfig

# Simulate AI-generated config (or use actual AI if key available)
ai_config = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(
            name='organizations',
            count=3,
            clear_before=True,
            columns=[
                ColumnConfig(name='org_code', generator='pattern', params={'pattern': ORG_PATTERN}),
                ColumnConfig(name='name', generator='company'),
                ColumnConfig(name='description', generator='sentence'),
            ]
        )
    ]
)

# Save and fill
config_path = Path('_ai_demo_config.yaml')
save_config(ai_config, str(config_path))
print('Generated YAML config:')
print(config_path.read_text()[:300])

results = fill_from_config(str(config_path), clear_before=True)
for r in results:
    print(f'\nFilled {r.table_name}: {r.count} rows in {r.elapsed:.3f}s')

config_path.unlink(missing_ok=True)



Generated YAML config:
db_path: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db
provider: mimesis
locale: en_US
tables:
- name: organizations
  count: 3
  batch_size: 5000
  columns:
  - name: org_code
    generator: pattern
    provider: null
    params:
      pattern: ORG-\d{4}
    null_ratio: 0.0
    d


Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]


Filled organizations: 3 rows in 0.032s


## 总结

| 功能 | 说明 |
|------|------|
| SchemaAnalyzer | 收集表上下文，发送给 LLM |
| AiConfigRefiner | 自纠正闭环：生成→验证→修复 |
| 模型选择 | 12 个免费模型，自动回退 |
| ErrorSummary | 7 种错误分类 |
| 文件缓存 | 平台标准缓存目录 |

**下一步**: [08-mcp-server.ipynb](08-mcp-server.ipynb) — MCP 服务器集成

In [8]:
# ✅ 验证: 确保数据已成功生成并写入
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # 基本行数验证
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
